# Notebook 02 — Usability Analysis

Analyses the **usability cost** of biometric security decisions — how often legitimate users are incorrectly rejected (FRR) and how threshold choices trade security (FAR) for user experience.

In access-control deployments, high FRR creates help-desk load, queue delays, and user workarounds — a direct usability failure even when FAR is low.

**Usability metrics in this notebook:**
- FRR at the EER operating point (primary usability indicator)
- Estimated false rejections per 1,000 authentication attempts
- Threshold sensitivity — how usability degrades as security tightens
- Algorithm comparison via DeLong AUC (ISO/IEC 19795-1:2021)


In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from biometric_auth.engine.metrics import run_evaluation, compute_far_frr_at_threshold
from biometric_auth.engine.statistics import delong_auc_comparison
from biometric_auth.parsers.score_file import load_scores

DATA_DIR = Path('../data/synthetic')
datasets = {
    'High (≈1% EER)':   load_scores(DATA_DIR / 'scores_high_accuracy.csv'),
    'Medium (≈5% EER)': load_scores(DATA_DIR / 'scores_medium_accuracy.csv'),
    'Low (≈15% EER)':   load_scores(DATA_DIR / 'scores_low_accuracy.csv'),
}

results = {name: run_evaluation(df, algorithm_name=name) for name, df in datasets.items()}
for name, r in results.items():
    print(f'{name}: FRR={r.frr:.3%}  FAR={r.far:.3%}  EER={r.eer:.3%}  AUC={r.auc_roc:.5f}')


In [ ]:
# Usability table: FRR and estimated false rejections per 1,000 attempts
DAILY_ATTEMPTS = 1000  # illustrative deployment volume
usability_rows = []
for name, r in results.items():
    false_rejects = int(round(r.frr * DAILY_ATTEMPTS))
    usability_rows.append({
        'System': name,
        'FRR at EER': f'{r.frr:.3%}',
        'FAR at EER': f'{r.far:.3%}',
        'False rejections / 1k attempts': false_rejects,
        'Usability grade': 'Good' if r.frr < 0.02 else ('Acceptable' if r.frr < 0.08 else 'Poor'),
    })
usability_df = pd.DataFrame(usability_rows)
usability_df


In [ ]:
# FRR comparison — primary usability chart
colours = ['#003087', '#007A73', '#C8102E']
names = list(results.keys())
frrs = [r.frr * 100 for r in results.values()]
frr_ci_lo = [(r.frr - r.frr_ci_low) * 100 for r in results.values()]
frr_ci_hi = [(r.frr_ci_high - r.frr) * 100 for r in results.values()]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(names, frrs, color=colours, alpha=0.85)
ax.errorbar(names, frrs, yerr=[frr_ci_lo, frr_ci_hi], fmt='none', color='black', capsize=6, lw=2)
ax.set_ylabel('FRR (%) — legitimate users rejected')
ax.set_title('Usability Cost: False Rejection Rate at EER Operating Point')
for bar, v in zip(bars, frrs):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.4, f'{v:.2f}%', ha='center', fontsize=10)
plt.tight_layout(); plt.show()


In [ ]:
# Usability–security trade-off: FAR vs FRR at multiple thresholds (medium-accuracy system)
medium_df = datasets['Medium (≈5% EER)']
thresholds = np.linspace(0.3, 0.9, 13)
tradeoff = []
for t in thresholds:
    far, frr = compute_far_frr_at_threshold(medium_df, float(t))
    tradeoff.append({'threshold': round(t, 2), 'FAR': far, 'FRR': frr})
tradeoff_df = pd.DataFrame(tradeoff)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tradeoff_df['threshold'], tradeoff_df['FRR'] * 100, 'o-', label='FRR (usability)', color='#003087', lw=2)
ax.plot(tradeoff_df['threshold'], tradeoff_df['FAR'] * 100, 's-', label='FAR (security)', color='#C8102E', lw=2)
ax.set_xlabel('Decision threshold')
ax.set_ylabel('Rate (%)')
ax.set_title('Usability–Security Trade-off (Medium-accuracy system)\nTighter thresholds improve security but reject more legitimate users')
ax.legend()
plt.tight_layout(); plt.show()
tradeoff_df.round(4)


In [ ]:
# ROC curves — usability read as TPR = 1 − FRR
fig, ax = plt.subplots(figsize=(7, 6))
for (name, r), c in zip(results.items(), colours):
    ax.plot(r.roc_fpr, r.roc_tpr, label=f'{name} (AUC={r.auc_roc:.4f})', color=c, lw=2)
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlabel('FAR (security risk)')
ax.set_ylabel('TPR = 1 − FRR (usability — genuine users accepted)')
ax.set_title('ROC Curves — Usability vs Security by System')
ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
# DeLong AUC comparison — are usability/security differences statistically significant?
names = list(datasets.keys())
dfs   = list(datasets.values())
print('DeLong AUC pairwise comparisons:')
for i in range(len(names)):
    for j in range(i+1, len(names)):
        z, p = delong_auc_comparison(dfs[i], dfs[j])
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        print(f'  {names[i]} vs {names[j]}: z={z:.3f}, p={p:.4e} {sig}')
